In [1]:
import os
from dotenv import load_dotenv
from youtube_transcript_api import YouTubeTranscriptApi , TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
load_dotenv()

C:\Users\piusd\AppData\Local\Temp\ipykernel_12112\1703417333.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


True

In [6]:
video_id = "J5_-l7WIO_w"

try:
    # Initialize the API object
    ytt_api = YouTubeTranscriptApi()

    # Fetch the transcript and convert to dictionary format
    fetched_transcript = ytt_api.fetch(video_id, languages=['hi'])
    transcript_list = fetched_transcript.to_raw_data()

    # Combine text items into a single string
    transcript = " ".join(chunk["text"] for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")
except Exception as e:
    print(f"An error occurred: {e}")

हाय गाइज़, माय नेम इज नितेश एंड यू आर वेलकम टू माय YouTube चैनल। इस वीडियो में भी हम लोग अपना लैंग चेन प्लेलिस्ट कंटिन्यू करेंगे। अह पिछले वीडियो में हमने रैग पढ़ना शुरू किया था और हमने फोकस किया था रैग के अराउंड जो भी थ्योरी है उसको डिस्कस करने के ऊपर। मैंने आपको बताया था कि रैग क्या होता है? उसकी जरूरत क्यों होती है? मैंने वहां पर रैक को कंपेयर करके भी दिखाया था फाइन ट्यूनिंग जैसी टेक्निक के साथ। और आज का जो वीडियो है वह पिछले वीडियो का ही कंटिन्यूएशन है। जहां पर हम प्रैक्टिकली एक रैग बेस्ड सिस्टम बनाएंगे यूजिंग लैंग चेन। प्लान यह है कि मैं एक प्रॉब्लम स्टेटमेंट उठाऊंगा और उस प्रॉब्लम स्टेटमेंट के अराउंड एक रैग बेस्ड सिस्टम क्रिएट करूंगा और यह सारा का सारा कोड हम लैंग चेन में करने वाले हैं। तो अभी तक आपने जो भी पढ़ा है पिछले चारप वीडियोस में डॉक्यूमेंट लोडर्स, टेक्स्ट स्प्लिटर्स, वेक्टर स्टर्स ये सब कुछ हम आज के वीडियो में यूज़ करेंगे और इनको यूज़ करके हम एक रैग बेस सिस्टम बनाएंगे। ऑन द होल इट्स गोइंग टू बी अ वेरी इंटरेस्टिंग वीडियो। लेट्स स्टार्ट। तो चलो गाइस, सबसे पहले बात करते हैं प्र

In [7]:
transcript_list

[{'text': 'हाय गाइज़, माय नेम इज नितेश एंड यू आर',
  'start': 0.0,
  'duration': 4.24},
 {'text': 'वेलकम टू माय YouTube चैनल। इस वीडियो में',
  'start': 2.0,
  'duration': 4.0},
 {'text': 'भी हम लोग अपना लैंग चेन प्लेलिस्ट',
  'start': 4.24,
  'duration': 3.92},
 {'text': 'कंटिन्यू करेंगे। अह पिछले वीडियो में',
  'start': 6.0,
  'duration': 5.12},
 {'text': 'हमने रैग पढ़ना शुरू किया था और हमने फोकस',
  'start': 8.16,
  'duration': 5.519},
 {'text': 'किया था रैग के अराउंड जो भी थ्योरी है',
  'start': 11.12,
  'duration': 5.2},
 {'text': 'उसको डिस्कस करने के ऊपर। मैंने आपको',
  'start': 13.679,
  'duration': 4.961},
 {'text': 'बताया था कि रैग क्या होता है? उसकी जरूरत',
  'start': 16.32,
  'duration': 4.799},
 {'text': 'क्यों होती है? मैंने वहां पर रैक को',
  'start': 18.64,
  'duration': 4.479},
 {'text': 'कंपेयर करके भी दिखाया था फाइन ट्यूनिंग',
  'start': 21.119,
  'duration': 4.561},
 {'text': 'जैसी टेक्निक के साथ। और आज का जो वीडियो',
  'start': 23.119,
  'duration': 4.881},
 {'text': 

In [8]:
# text splitter
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000 , chunk_overlap = 200)
chunk = splitter.create_documents([transcript])

In [10]:
chunk[:5]

[Document(metadata={}, page_content='हाय गाइज़, माय नेम इज नितेश एंड यू आर वेलकम टू माय YouTube चैनल। इस वीडियो में भी हम लोग अपना लैंग चेन प्लेलिस्ट कंटिन्यू करेंगे। अह पिछले वीडियो में हमने रैग पढ़ना शुरू किया था और हमने फोकस किया था रैग के अराउंड जो भी थ्योरी है उसको डिस्कस करने के ऊपर। मैंने आपको बताया था कि रैग क्या होता है? उसकी जरूरत क्यों होती है? मैंने वहां पर रैक को कंपेयर करके भी दिखाया था फाइन ट्यूनिंग जैसी टेक्निक के साथ। और आज का जो वीडियो है वह पिछले वीडियो का ही कंटिन्यूएशन है। जहां पर हम प्रैक्टिकली एक रैग बेस्ड सिस्टम बनाएंगे यूजिंग लैंग चेन। प्लान यह है कि मैं एक प्रॉब्लम स्टेटमेंट उठाऊंगा और उस प्रॉब्लम स्टेटमेंट के अराउंड एक रैग बेस्ड सिस्टम क्रिएट करूंगा और यह सारा का सारा कोड हम लैंग चेन में करने वाले हैं। तो अभी तक आपने जो भी पढ़ा है पिछले चारप वीडियोस में डॉक्यूमेंट लोडर्स, टेक्स्ट स्प्लिटर्स, वेक्टर स्टर्स ये सब कुछ हम आज के वीडियो में यूज़ करेंगे और इनको यूज़ करके हम एक रैग बेस सिस्टम बनाएंगे। ऑन द होल इट्स गोइंग टू बी अ वेरी इंटरेस्टिंग वीडियो। लेट्स स्टार्ट। तो

In [11]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
vector_store = Chroma.from_documents(
     chunk,
     embedding=embeddings
)

In [14]:
vector_store._collection.get()["ids"]

['5508bf75-f5fe-4aec-902e-6e0099eab090',
 'e7f59fa9-7c76-45f2-aba3-c14f0acb95de',
 '7930b9a7-9202-4c01-b8da-1db74da25a43',
 '11bb6d51-068d-443c-adc1-6629763de689',
 'bb128654-0e7d-4fb6-8d92-0f6c06338d24',
 '9752dede-6e31-459a-9363-3c3cffa3f875',
 'de2301b2-ff56-4793-b3d5-a4c72b864e68',
 '1ea95d6e-9fee-4161-92a3-ac01d9bd3fde',
 'a1ceaaa5-bb2b-41aa-abe1-9e7ab1e885a9',
 '0969feb9-4cc9-46ac-9df2-a85c01575001',
 '58b0d1ae-59e3-435a-aa69-adc6cf58342e',
 '6d81d09a-64a4-492b-85c4-e966844e5668',
 '83d3a3cf-9a04-4d6a-994a-0d49b89c99bd',
 '3706992b-341d-4a1f-93a3-d699e1db0b1f',
 '49a9903a-5fcc-4a49-9f25-5eb4d2015591',
 'c3ed4c79-93f2-495f-9864-19d398113945',
 '0f34a524-327b-490d-9ca1-501eb7964fd3',
 '28959e3e-ee66-4ab1-a50a-624a4f91b797',
 'c715156b-557e-4347-9897-e1e04414fc12',
 'd755f116-0bea-408b-8ca1-f610eae3a662',
 '28dce76b-f95f-49fa-9039-4e4aae0e58c4',
 '90b0b3a9-b2f1-473e-92f1-bcf5bc14cbcc',
 'aaf85c6c-1854-46d7-b86b-5eeecf0458a7',
 '09064bcf-3441-4136-b038-2b4f75699fd6',
 'c187e602-777a-

In [15]:
vector_store._collection.get(ids=['fcdf3c34-3393-4347-ac59-fc37cae2a199'])

{'ids': ['fcdf3c34-3393-4347-ac59-fc37cae2a199'],
 'embeddings': None,
 'documents': ['मेरे रैक सिस्टम को याद है कि मैंने एक हफ्ते पहले भी उससे क्या बात की थी। तो वो कॉन्टेक्स्ट पकड़ करके मुझे आंसर कर रहा है कि देखो एक हफ्ते पहले भी जब हमारी बात हुई थी तो हमने इस बारे में यह बात की थी। तो मेमोरी बेस्ड एप्लीकेशनेशंस भी आप बना सकते हो। सो ऑन द होल मैं बस आपको यह बताना चाह रहा हूं कि हमने पिछले वीडियो और आज के वीडियो में जो भी चीजें पढ़ी वो सिर्फ सरफेस था रैग का। रैग खुद में एक बहुत बड़ी और बहुत पावरफुल चीज है। और जब आप एक प्रॉपर इंडस्ट्री ग्रेड रैग सिस्टम बनाते हो तो आप बहुत तरह की प्रॉब्लम्स फेस करोगे और उन प्रॉब्लम्स को सॉल्व करने के लिए बहुत तरह की टेक्निक्स एग्जिस्ट करती हैं जैसा मैंने आपको बताया। तो अभी इंडस्ट्री में एक कंप्लीटली नई फील्ड बन के उभर के आई है जिसका नाम है एडवांस्ड रैग। ठीक है? बहुत सारे स्टूडेंट्स मुझे कमेंट्स में लिख के बता रहे थे कि सर आप ये सारी टेक्निक्स कब पढ़ाओगे? तो मैं बस क्लेरिफाई करना चाहता हूं कि ये जो टेक्निक्स मैंने आपको अभी 10-15 मिनट में बताई हैं ये हम ल